**Name:** Keerthi N  
**Registration Number:** 2411021240024  
**Course:** Generative AI


# Assignment 11: Fine-Tuning GPT-2 on a Custom Text Style

## Objective
Understand how fine-tuning helps a pretrained GPT-2 language model adapt to a specific writing style.

## Workflow
1. Create/load a 200–300-line custom dataset.
2. Clean the dataset.
3. Load original GPT-2 and generate baseline text using the Assignment 2 prompt.
4. Tokenize the dataset.
5. Fine-tune GPT-2 using Hugging Face Trainer.
6. Generate text using the fine-tuned model.
7. Compare both outputs and save the required files.

**Style chosen:** original fictional mystery writing.

## 1. Install required libraries

In [ ]:
!pip install -q transformers accelerate

## 2. Import libraries

In [ ]:
import os
import re
import random
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
GPU available: False


## 3. Load the included dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload custom_mystery_dataset.txt from this ZIP if prompted

dataset_file = "custom_mystery_dataset.txt"
with open(dataset_file, "r", encoding="utf-8") as f:
    dataset_lines = f.read().splitlines()

print("Dataset lines:", len(dataset_lines))
print("\nFirst five examples:")
for line in dataset_lines[:5]:
    print(line)

Saving custom_mystery_dataset.txt to custom_mystery_dataset (1).txt
Dataset lines: 240

First five examples:
The night guard inside the forgotten research room saw a message appearing on the computer screen, and felt that something was watching nearby.
The brave student inside the underground chamber noticed that every clock had stopped at midnight, while the cold air moved through the corridor.
The night guard inside the abandoned laboratory noticed a strange shadow moving across the floor, and immediately wrote down every detail.
The curious researcher in the empty science building discovered a key covered with unusual markings, but kept the discovery secret until morning.
The night guard next to the silent machine saw a message appearing on the computer screen, and immediately wrote down every detail.


## 4. Clean the dataset

In [ ]:
cleaned_lines = []
for line in dataset_lines:
    line = line.strip()
    line = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\b", "", line)
    line = re.sub(r"@\w+", "", line)
    line = re.sub(r"\s+", " ", line).strip()
    if line:
        cleaned_lines.append(line)

cleaned_lines = list(dict.fromkeys(cleaned_lines))
with open("cleaned_mystery_dataset.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned_lines))

print("Before cleaning:", len(dataset_lines))
print("After cleaning:", len(cleaned_lines))
assert 200 <= len(cleaned_lines) <= 300, "Dataset should contain 200–300 lines."

Before cleaning: 240
After cleaning: 240


## 5. Load original GPT-2

In [ ]:
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained(model_name)
base_model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = base_model.to(device)
print("Original GPT-2 loaded on:", device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Original GPT-2 loaded on: cpu


## 6. Using the exact prompt from Assignment 2



In [ ]:
prompt = "The young scientist entered the abandoned laboratory and discovered"

def generate_text(model, prompt, seed=42):
    model.eval()
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

base_output = generate_text(base_model, prompt)
print("ORIGINAL GPT-2 OUTPUT")
print(base_output)

with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write(base_output)

ORIGINAL GPT-2 OUTPUT
The young scientist entered the abandoned laboratory and discovered that there was a large, humanoid body lying on its back. The corpse's name would be revealed later in this story as "Giovanni".
A new dimension appeared within Gio who attempted to use his powers against him but could not stop it from entering into reality.[9] He used an energy shield with which he destroyed many of their lives before they were caught by Cocytus


## 7. Prepare the dataset for training

In [ ]:
from torch.utils.data import Dataset

class MysteryDataset(Dataset):
    def __init__(self, lines, tokenizer, max_length=96):
        self.examples = []
        for line in lines:
            encoded = tokenizer(
                line,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )
            self.examples.append({
                "input_ids": encoded["input_ids"].squeeze(0),
                "attention_mask": encoded["attention_mask"].squeeze(0)
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

train_dataset = MysteryDataset(cleaned_lines, tokenizer)
print("Training examples:", len(train_dataset))

Training examples: 240


## 8. Fine-tune GPT-2 using Hugging Face Trainer

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.config.pad_token_id = tokenizer.eos_token_id

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2_mystery_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

trainer.train()
print("Fine-tuning completed!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,3.258207
40,1.484741
60,0.898742
80,0.690226
100,0.591717
120,0.534697
140,0.509823
160,0.486703
180,0.509605
200,0.501017


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning completed!


## 9. Save the fine-tuned model checkpoint

In [ ]:
save_path = "./fine_tuned_gpt2_mystery"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Checkpoint saved:", save_path)
print(os.listdir(save_path))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved: ./fine_tuned_gpt2_mystery
['tokenizer.json', 'generation_config.json', 'model.safetensors', 'config.json', 'tokenizer_config.json', 'training_args.bin']


## 10. Generate text using the fine-tuned model

In [ ]:
fine_tuned_model = GPT2LMHeadModel.from_pretrained(save_path).to(device)
fine_tuned_output = generate_text(fine_tuned_model, prompt)

print("FINE-TUNED GPT-2 OUTPUT")
print(fine_tuned_output)

with open("fine_tuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write(fine_tuned_output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

FINE-TUNED GPT-2 OUTPUT
The young scientist entered the abandoned laboratory and discovered a notebook filled with impossible equations, while an alarm echoed through the building.
"It's been three hours," she said as the machine suddenly began to vibrate. "Suddenly felt like something was watching nearby." Suddenly noticed that every clock had stopped at midnight, although nobody else believed what happened next morning.[src]

 (Image source: Wikipedia)
-Doctor inside abandoned research room - unknown


## 11. Compare original and fine-tuned outputs

In [ ]:
print("=" * 60)
print("PROMPT")
print(prompt)
print("\n" + "=" * 60)
print("ORIGINAL GPT-2")
print(base_output)
print("\n" + "=" * 60)
print("FINE-TUNED GPT-2")
print(fine_tuned_output)

PROMPT
The young scientist entered the abandoned laboratory and discovered

ORIGINAL GPT-2
The young scientist entered the abandoned laboratory and discovered that there was a large, humanoid body lying on its back. The corpse's name would be revealed later in this story as "Giovanni".
A new dimension appeared within Gio who attempted to use his powers against him but could not stop it from entering into reality.[9] He used an energy shield with which he destroyed many of their lives before they were caught by Cocytus

FINE-TUNED GPT-2
The young scientist entered the abandoned laboratory and discovered a notebook filled with impossible equations, while an alarm echoed through the building.
"It's been three hours," she said as the machine suddenly began to vibrate. "Suddenly felt like something was watching nearby." Suddenly noticed that every clock had stopped at midnight, although nobody else believed what happened next morning.[src]

 (Image source: Wikipedia)
-Doctor inside abandone

### Before-and-after analysis

1. **Vocabulary:**
The original GPT-2 output included fantasy-related words and ideas such as a humanoid body, dimensions, and energy shields. The fine-tuned output included mystery and laboratory-related words and ideas such as a notebook, impossible equations, an alarm, and a machine. This shows that the fine-tuned output used vocabulary more closely related to the mystery and laboratory theme.

2. **Theme consistency or sentence patterns:**
The original output moved toward fantasy ideas, including energy shields and another dimension. The fine-tuned output focused more on strange discoveries, a laboratory, and a machine. However, the fine-tuned output repeated words such as "suddenly" and contained some confusing sentences. This shows that fine-tuning influenced the theme, but the generated text was not fully consistent or coherent.

### Conclusion
In this assignment, a pretrained GPT-2 model was fine-tuned on a custom mystery-writing dataset using Hugging Face Trainer. Text was generated with the original and fine-tuned models using the same prompt and generation settings. Their outputs were compared to investigate changes in vocabulary, theme consistency, sentence patterns and repetition. The degree of adaptation depends on dataset quality, size and variety.